# AI Video Studio — Colab Qwen Model Lab

**Purpose:** temporary, interactive GPU smoke-test for the Qwen/Ollama planning path. This notebook is a model experiment, not a Studio deployment.

## Scope and safety

- Starts a temporary Ollama daemon bound to loopback (127.0.0.1) only for interactive notebook requests; it does not expose a public service or open a tunnel.
- Installs Ollama inside the current Colab runtime and downloads only the selected model tag.
- Sends one bounded Arabic planning request and validates the returned JSON structure and scene-duration total.
- Saves a small evidence report. Model weights remain in the ephemeral Colab VM and are not copied to Drive.
- Does not upload .env, API keys, user media, or Studio project data.
- Does not run a persistent Studio UI, external API service, Celery worker, or distributed worker.

**Free-tier limitation:** Colab does not guarantee GPU availability or a fixed runtime allowance. Sessions can be terminated and the VM is temporary. Use this notebook interactively and repeat an interrupted test as needed. See the [official Colab FAQ](https://research.google.com/colaboratory/faq.html).

## Before running

1. Open Runtime → Change runtime type → GPU.
2. Run cells from top to bottom.
3. If GPU is unavailable, stop before pulling model weights and retry when a GPU runtime is offered.
4. The default model is qwen3.5:0.8b (approximately 1.2–1.3 GB in the Ollama library). Change MODEL_TAG only after checking the model's current size and license.




In [ ]:
# 1. Configuration and GPU preflight

import json
import os
import platform
import shutil
import subprocess
import time
from pathlib import Path

import requests
import torch

LAB_ROOT = Path("/content/ai-video-studio-model-lab")
MODEL_DIR = LAB_ROOT / "ollama-models"
REPORT_DIR = LAB_ROOT / "reports"
LOG_DIR = LAB_ROOT / "logs"

for directory in (MODEL_DIR, REPORT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_TAG = os.environ.get("AI_STUDIO_OLLAMA_MODEL", "qwen3.5:0.8b")
OLLAMA_HOST = "127.0.0.1:11434"

os.environ["OLLAMA_MODELS"] = str(MODEL_DIR)
os.environ["OLLAMA_HOST"] = OLLAMA_HOST

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached to this Colab session. "
        "Select Runtime > Change runtime type > GPU, reconnect, and rerun."
    )

gpu = torch.cuda.get_device_properties(0)
gpu_info = {
    "name": gpu.name,
    "total_vram_gib": round(gpu.total_memory / (1024 ** 3), 2),
    "cuda_version": torch.version.cuda,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
}

print(json.dumps(gpu_info, indent=2, ensure_ascii=False))
GPU_BEFORE = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu,temperature.gpu", "--format=csv,noheader"],
    check=True, text=True, capture_output=True
).stdout.strip()
print(GPU_BEFORE)
if gpu.total_memory < 4 * (1024 ** 3):
    raise RuntimeError(
        "The assigned GPU exposes less than 4 GiB VRAM. "
        "Stop before downloading weights; this notebook's Qwen GPU test is not approved for this device."
    )




In [ ]:
# 2. Install and start Ollama locally

# The official Linux archive is installed only when the CLI is absent. No systemd service or public tunnel is created.

import subprocess
import requests
import time

if shutil.which("ollama") is None:
    archive = Path("/tmp/ollama-linux-amd64.tar.zst")
    subprocess.run(
        [
            "bash", "-lc",
            "curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst "
            "-o /tmp/ollama-linux-amd64.tar.zst",
        ],
        check=True,
        text=True,
    )
    if shutil.which("zstd") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    subprocess.run(
        ["tar", "--zstd", "-xf", str(archive), "-C", "/usr"],
        check=True,
    )
    archive.unlink(missing_ok=True)

OLLAMA_VERSION = subprocess.run(
    ["ollama", "--version"],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()
print(OLLAMA_VERSION)

def ollama_ready():
    try:
        response = requests.get(f"http://{OLLAMA_HOST}/api/tags", timeout=3)
        return response.ok
    except requests.RequestException:
        return False

OLLAMA_PROCESS = None
OLLAMA_LOG_HANDLE = None

if not ollama_ready():
    server_env = os.environ.copy()
    server_env["OLLAMA_HOST"] = OLLAMA_HOST
    server_env["OLLAMA_MODELS"] = str(MODEL_DIR)
    OLLAMA_LOG_HANDLE = open(LOG_DIR / "ollama-serve.log", "w", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        env=server_env,
        stdout=OLLAMA_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

deadline = time.time() + 90
while time.time() < deadline and not ollama_ready():
    if OLLAMA_PROCESS is not None and OLLAMA_PROCESS.poll() is not None:
        raise RuntimeError(
            "Ollama server exited during startup. Inspect the runtime log under the lab directory."
        )
    time.sleep(2)

if not ollama_ready():
    raise TimeoutError("Ollama did not become ready on loopback within 90 seconds.")

print("Ollama API is ready on loopback:", OLLAMA_HOST)







In [ ]:
# 3. Pull the selected model

# Only the configured model tag is downloaded. The model cache is under the temporary Colab VM, not Google Drive.

print("Selected model:", MODEL_TAG)
print("Model cache:", MODEL_DIR)

existing = requests.get(f"http://{OLLAMA_HOST}/api/tags", timeout=10).json()
installed_tags = {item.get("name") for item in existing.get("models", [])}

if MODEL_TAG not in installed_tags:
    subprocess.run(
        ["ollama", "pull", MODEL_TAG],
        check=True,
        text=True,
    )
else:
    print("Model already exists in this runtime.")

models = requests.get(f"http://{OLLAMA_HOST}/api/tags", timeout=10).json()
matching = [
    {key: model.get(key) for key in ("name", "size", "digest", "modified_at")}
    for model in models.get("models", [])
    if model.get("name") == MODEL_TAG
]
if not matching:
    raise RuntimeError(f"Ollama did not register the expected model tag: {MODEL_TAG}")

print(json.dumps(matching[0], indent=2, ensure_ascii=False))



In [ ]:
# 4. Arabic CreativePlan smoke test

# The test asks for exactly three five-second scenes (15 seconds total). The response is parsed as JSON and checked for required fields before it is accepted as a Qwen planning pass.

import json
import time

SYSTEM_PROMPT = """You are the executive video director inside AI Video Studio.
Return one valid JSON object only. Do not use Markdown fences or add prose outside JSON.
All user-facing textual values must be in Arabic.
The JSON must contain:
title, logline, target_duration_seconds, audience, visual_style,
color_palette, audio_direction, scenes.
Each scenes item must contain:
index, duration_seconds, purpose, visual_description, shot_type,
camera_movement, narration, on_screen_text, sound_design.
"""

USER_PROMPT = """أنشئ خطة إنتاج عربية لإعلان سينمائي قصير لمنتج فاخر غير محدد الاسم.
المطلوب بالضبط ثلاثة مشاهد، مدة كل مشهد 5 ثوانٍ، والإجمالي 15 ثانية.
اجعل المشاهد مترابطة بصرياً، ولا تخترع اسماً أو شعاراً أو مواصفات لعلامة تجارية.
أعد JSON فقط، والتزم بالحقول المطلوبة في تعليمات النظام.
"""

payload = {
    "model": MODEL_TAG,
    "stream": False,
    "keep_alive": "5m",
    "format": "json",
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    "options": {
        "temperature": 0.1,
        "num_ctx": 4096,
        "num_predict": 1400,
    },
}

started = time.perf_counter()
response = requests.post(
    f"http://{OLLAMA_HOST}/api/chat",
    json=payload,
    timeout=300,
)
response.raise_for_status()
result = response.json()
elapsed_seconds = round(time.perf_counter() - started, 3)

content = result.get("message", {}).get("content", "")
if not content:
    raise RuntimeError("Ollama returned an empty model response.")

plan = json.loads(content)

required_top_level = {
    "title", "logline", "target_duration_seconds", "audience",
    "visual_style", "color_palette", "audio_direction", "scenes",
}
required_scene_fields = {
    "index", "duration_seconds", "purpose", "visual_description",
    "shot_type", "camera_movement", "narration", "on_screen_text",
    "sound_design",
}

missing_top = required_top_level - set(plan)
assert not missing_top, f"Missing CreativePlan fields: {sorted(missing_top)}"
assert isinstance(plan["scenes"], list) and len(plan["scenes"]) == 3
assert int(plan["target_duration_seconds"]) == 15
assert isinstance(plan["color_palette"], list)
assert 2 <= len(plan["color_palette"]) <= 6

def contains_arabic(value):
    return isinstance(value, str) and any(0x0600 <= ord(char) <= 0x06FF for char in value)

for field in ("title", "logline", "visual_style", "audio_direction"):
    assert contains_arabic(plan[field]), f"{field} does not contain Arabic text."

for expected_index, scene in enumerate(plan["scenes"], start=1):
    assert int(scene["index"]) == expected_index, "Scene indexes must be sequential from 1."
    missing_scene = required_scene_fields - set(scene)
    assert not missing_scene, f"Scene missing fields: {sorted(missing_scene)}"
    assert isinstance(scene["duration_seconds"], (int, float))
    assert int(scene["duration_seconds"]) == 5

actual_duration = sum(int(scene["duration_seconds"]) for scene in plan["scenes"])
assert actual_duration == 15, f"Scene duration sum is {actual_duration}, expected 15."

print("PASS — valid Arabic CreativePlan JSON")
print("Elapsed seconds:", elapsed_seconds)
print(json.dumps(plan, indent=2, ensure_ascii=False))






In [ ]:
# 5. Capture runtime and GPU evidence

# The report includes model digest, runtime timing, model response validity, and GPU telemetry. It deliberately does not contain API tokens, private project details, or source media.

def command_output(command):
    completed = subprocess.run(command, text=True, capture_output=True, check=False)
    return {
        "returncode": completed.returncode,
        "stdout": completed.stdout.strip()[-8000:],
        "stderr": completed.stderr.strip()[-2000:],
    }

ollama_ps = command_output(["ollama", "ps"])
gpu_offload_observed = any(
    "GPU" in line.upper()
    for line in ollama_ps["stdout"].splitlines()[1:]
)
if not gpu_offload_observed:
    raise RuntimeError(
        "Ollama did not report GPU offload for the loaded model. "
        "Do not mark this run as a GPU acceptance pass."
    )

gpu_after = command_output([
    "nvidia-smi",
    "--query-gpu=name,memory.total,memory.used,utilization.gpu,temperature.gpu",
    "--format=csv,noheader",
])

timing = {
    key: result.get(key)
    for key in (
        "total_duration", "load_duration", "prompt_eval_count",
        "prompt_eval_duration", "eval_count", "eval_duration",
    )
}

report = {
    "schema_version": 1,
    "product": "AI Video Studio",
    "test_name": "colab_qwen_arabic_creative_plan_smoke",
    "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "status": "passed",
    "model": {
        "tag": MODEL_TAG,
        "registry_record": matching[0],
    },
    "runtime": gpu_info,
    "ollama_version": OLLAMA_VERSION,
    "gpu_before": GPU_BEFORE,
    "inference": {
        "elapsed_seconds": elapsed_seconds,
        "gpu_offload_observed": gpu_offload_observed,
        "ollama_timing_nanoseconds": timing,
        "creative_plan_valid": True,
        "scene_count": len(plan["scenes"]),
        "target_duration_seconds": plan["target_duration_seconds"],
        "scene_duration_sum_seconds": actual_duration,
    },
    "gpu_after": gpu_after,
    "ollama_process_snapshot": ollama_ps,
    "limitations": [
        "Interactive Colab smoke test only; not a persistent Studio runtime.",
        "This does not validate Studio API integration or ComfyUI workflows.",
        "A successful Qwen smoke test does not certify video generation quality.",
        "Colab free GPU availability and runtime lifetime are not guaranteed.",
    ],
}

report_path = REPORT_DIR / "qwen-arabic-smoke-report.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Evidence report saved:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))





In [ ]:
# 6. Optional: copy only the report to Google Drive

# This is optional. The notebook does not copy model weights or private project data into Drive.

from google.colab import drive
drive.mount("/content/drive")

drive_report_dir = Path("/content/drive/MyDrive/AI-Video-Studio/Colab-Reports")
drive_report_dir.mkdir(parents=True, exist_ok=True)
destination = drive_report_dir / report_path.name
shutil.copy2(report_path, destination)

print("Evidence report copied to:", destination)




## Acceptance boundary

A Qwen pass means only that this model produced a valid, duration-consistent Arabic CreativePlan on the reported Colab GPU.

It does not mean:
- Qwen has been connected to the Studio API on the Windows host.
- Wan 2.2, RIFE, Real-ESRGAN, ComfyUI, or FFmpeg video rendering passed.
- The Colab free tier can host the Studio, a public endpoint, or a persistent worker.
- The resulting creative plan has been accepted for production quality.

Attach the generated JSON report to the model-acceptance record, review the actual plan, and repeat the test whenever the model tag, quantization, runtime, or prompt contract changes.
